In [1]:
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path("../src").resolve()))

from exoplanet_detector.models.train import (
    build_search_summary,
    get_default_model_specs,
    get_default_scoring,
    run_model_searches,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)


In [2]:
KOI_train_set = pd.read_csv("../data/processed/KOI_train_set.csv")
KOI_test_set = pd.read_csv("../data/processed/KOI_test_set.csv")
K2P_set = pd.read_csv("../data/processed/K2P_set.csv")

The full preprocessing pipeline was implemented and imported from `pipelines.py`. Now, 5 different models will be trained and tuned and the one performing the best will be chosen.

In [3]:
X_train = KOI_train_set.drop(columns=["label", "group_id"])
y_train = KOI_train_set["label"]
group_id = KOI_train_set["group_id"]

In [4]:
X_test = KOI_test_set.drop(columns=["label", "group_id"])
y_test = KOI_test_set["label"]

In [5]:
X_k2p = K2P_set.drop(columns=["label", "group_id"])
y_k2p = K2P_set["label"]

In [6]:
model_specs = get_default_model_specs()

In [7]:
RUN_TAG = "v1"  # bump when you intentionally change search space/pipeline
FORCE_RETRAIN = False

ARTIFACT_DIR = Path("../artifacts/model_search") / RUN_TAG
SEARCH_RESULTS_PATH = ARTIFACT_DIR / "search_results.joblib"
SUMMARY_PATH = ARTIFACT_DIR / "cv_summary.csv"

For exoplanets, recall is the important metric, since missing actual planets is more costly than exaggerating their number. Nevertheless, we still want precision to be high. For this reason, we will try to maximize f2, which will weight recall 4 times as much as precision.

In [8]:
scoring = get_default_scoring()
print(scoring)

{'recall': 'recall', 'precision': 'precision', 'f2': make_scorer(fbeta_score, response_method='predict', beta=2), 'pr_auc': 'average_precision'}


In [9]:
search_results = run_model_searches(
    X_train,
    y_train,
    group_id,
    model_specs=model_specs,
    scoring=scoring,
    n_splits=5,
    cv_random_state=42,
    n_iter=25,
    n_jobs=-1,
    search_random_state=42,
    refit_metric="f2",
    verbose=1,
    artifact_path=SEARCH_RESULTS_PATH,
    force_retrain=FORCE_RETRAIN,
)

summary_df = build_search_summary(search_results)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(SUMMARY_PATH, index=False)
print(f"Saved summary to: {SUMMARY_PATH}")

Saved summary to: ../artifacts/model_search/v1/cv_summary.csv


In [10]:
summary_df = build_search_summary(search_results)

summary_df

,model,best_score_refit_f2,best_mean_test_recall,best_mean_test_precision,best_mean_test_f2,best_mean_test_pr_auc,best_params
0,svc_rbf,0.916118,0.944924,0.816951,0.916118,0.921851,"{'clf__C': 72.86653737491046, 'clf__gamma': 0.026619018884890575}"
1,rf,0.900404,0.925349,0.813167,0.900404,0.923146,"{'clf__max_depth': 10, 'clf__n_estimators': 220}"
2,knn,0.899316,0.922166,0.818403,0.899316,0.915006,"{'clf__n_neighbors': 30, 'clf__weights': 'distance'}"
3,hgb,0.889750,0.898952,0.855415,0.889750,0.931531,"{'clf__learning_rate': 0.052466345336252856, 'clf__max_leaf_nodes': 21}"
4,logreg,0.873978,0.919436,0.729929,0.873978,0.810868,{'clf__C': 4.5705630998014515}
